In [1]:
!pip uninstall transformers -y
!pip install transformers==4.50.0

Found existing installation: transformers 4.50.0
Uninstalling transformers-4.50.0:
  Successfully uninstalled transformers-4.50.0
  Using cached transformers-4.50.0-py3-none-any.whl.metadata (39 kB)
Using cached transformers-4.50.0-py3-none-any.whl (10.2 MB)


In [2]:
import transformers
print(transformers.__version__)

4.50.0


In [3]:
from transformers import T5Tokenizer,TFT5ForConditionalGeneration
import pandas as pd
import tensorflow as tf

In [4]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")
model = TFT5ForConditionalGeneration.from_pretrained("t5-small")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

All PyTorch model weights were used when initializing TFT5ForConditionalGeneration.

All the weights of TFT5ForConditionalGeneration were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFT5ForConditionalGeneration for predictions without further training.


In [5]:
data = pd.read_csv(r"/content/sample_data/IMDB Dataset.csv")
print(data.head(5))
data = data.groupby("sentiment").sample(
    n=2500,
    random_state=42
).reset_index(drop=True)
print(data["sentiment"].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
negative    2500
positive    2500
Name: count, dtype: int64


In [6]:
inputs = "sentiment: " + data["review"]
label = data["sentiment"]

In [10]:
input_tokens = tokenizer(inputs.tolist(),max_length=256,truncation=True,padding=True,return_tensors="tf")
label_tokens = tokenizer(label.tolist(),max_length=10,truncation=True,padding=True,return_tensors="tf")

In [11]:
labels_id = label_tokens["input_ids"]
labels_id = tf.where(labels_id==tokenizer.pad_token_id,-100,labels_id)

In [12]:
dataset = tf.data.Dataset.from_tensor_slices({
    "input_ids":input_tokens["input_ids"],
    "attention_mask":input_tokens["attention_mask"],
    "labels":labels_id
})
dataset = dataset.shuffle(
    buffer_size=len(data),
    seed=42
).batch(2)

In [13]:
optimizer = tf.keras.optimizers.Adam(learning_rate=3e-4)
model.compile(optimizer=optimizer)

In [14]:
model.fit(dataset,epochs=2)

Epoch 1/2
2500/2500 [==============================] - 260s 85ms/step - loss: 0.1902
Epoch 2/2
2500/2500 [==============================] - 212s 85ms/step - loss: 0.1218


In [15]:
test_sentences = [
    "sentiment: This movie was absolutely terrible and boring.",
    "sentiment: I hated this movie. It was awful.",
    "sentiment: This movie was fantastic and amazing.",
    "sentiment: I loved this movie. It was excellent."
]

for text in test_sentences:
    test_input = tokenizer(text, return_tensors="tf")

    output = model.generate(
        input_ids=test_input["input_ids"],
        attention_mask=test_input["attention_mask"],
        max_length=10
    )

    result = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    print(text)
    print("Prediction:", result)
    print()

sentiment: This movie was absolutely terrible and boring.
Prediction: negative

sentiment: I hated this movie. It was awful.
Prediction: negative

sentiment: This movie was fantastic and amazing.
Prediction: positive

sentiment: I loved this movie. It was excellent.
Prediction: positive

